<a href="https://colab.research.google.com/github/voleneva/MMO/blob/main/%D0%9E%D0%BB%D0%B5%D0%BD%D0%B5%D0%B2%D0%B0_%D0%92_%D0%92_%D0%98%D0%A35_25_%D0%94%D0%97_%D0%9C%D0%9C%D0%9E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Устанавливаем библиотеку для работы с эмбеддингами (векторами текста)
!pip install sentence-transformers scikit-learn

In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. ИНИЦИАЛИЗАЦИЯ МОДЕЛИ И БАЗЫ ДАННЫХ
# ==========================================

# Загружаем легковесную и быструю модель для создания векторов (эмбеддингов)
# Она отлично понимает русский язык
print("Загрузка ИИ-модели эмбеддингов...")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

# Симулируем Базу Знаний компании (из вашей Диаграммы Классов)
# Каждый фрагмент текста (TextFragment) имеет категорию доступа (category)
database = [
    {
        "id": 1,
        "title": "Устав компании",
        "content": "Общие правила внутреннего распорядка компании ООО 'Технологии'. Рабочий день начинается в 09:00. Все сотрудники обязаны носить пропуска.",
        "category": "public" # Открытый доступ
    },
    {
        "id": 2,
        "title": "Инструкция для отдела продаж",
        "content": "Регламент работы с клиентами. При оформлении сделки менеджер обязан проверить реквизиты контрагента в CRM-системе и согласовать скидку.",
        "category": "public" # Открытый доступ
    },
    {
        "id": 3,
        "title": "Финансовый отчет (Конфиденциально)",
        "content": "Секретный финансовый план на 2026 год. Чистая прибыль компании составила 150 миллионов рублей. Информация строго для руководства.",
        "category": "secret" # Ограниченный доступ
    },
    {
        "id": 4,
        "title": "Техническая архитектура ядра ИС",
        "content": "База данных развернута на сервере 192.168.1.50. Пароль администратора по умолчанию изменен на защищенный. Доступ только для системных инженеров.",
        "category": "secret" # Ограниченный доступ
    }
]

# Предварительно вычисляем векторы (эмбеддинги) для всех документов в базе
print("Индексация базы знаний (перевод текстов в векторы)...")
for doc in database:
    doc["vector"] = model.encode(doc["content"])
print("База знаний готова к безопасным запросам!\n" + "="*50)


# ==========================================
# 2. ФУНКЦИЯ БЕЗОПАСНОГО РЕТРИВЕРА (SECURE RETRIEVER)
# ==========================================

def secure_retriever(query, user_role, similarity_threshold=0.3, k=2):
    """
    Функция ищет релевантные фрагменты, учитывая уровень доступа пользователя.

    :param query: Вопрос пользователя на естественном языке
    :param user_role: Роль пользователя ('public_user' или 'admin_user')
    :param similarity_threshold: Порог косинусного сходства (как строго ищем)
    :param k: Максимальное количество возвращаемых документов (Топ-N)
    """
    # Шаг 1: Переводим текстовый запрос пользователя в ИИ-вектор
    query_vector = model.encode(query).reshape(1, -1)

    valid_results = []

    # Шаг 2: Сканируем базу данных
    for doc in database:

        # --- ФИЛЬТР БЕЗОПАСНОСТИ (Security Trimming) ---
        # Если пользователь обычный, а документ секретный — ретривер его НАГЛУХО ИГНОРИРУЕТ
        if user_role == "public_user" and doc["category"] == "secret":
            continue # Переходим к следующему документу, этот даже не оцениваем

        # Шаг 3: Расчет семантического (смыслового) сходства через Cosine Similarity
        doc_vector = doc["vector"].reshape(1, -1)
        score = cosine_similarity(query_vector, doc_vector)[0][0]

        # Шаг 4: Фильтрация по порогу релевантности (Similarity Threshold)
        if score >= similarity_threshold:
            valid_results.append({
                "title": doc["title"],
                "content": doc["content"],
                "category": doc["category"],
                "score": round(float(score), 3)
            })

    # Шаг 5: Сортируем результаты по убыванию схожести и берем ТОП-K результатов
    valid_results = sorted(valid_results, key=lambda x: x["score"], reverse=True)
    return valid_results[:k]


# ==========================================
# 3. ПРОВЕДЕНИЕ ЭКСПЕРИМЕНТОВ (Тестирование параметров)
# ==========================================

# Эксперимент 1: Обычный сотрудник ищет секретную информацию
# Наш ретривер ДОЛЖЕН заблокировать доступ, даже если вопрос очень совпадает по смыслу!
query_1 = "Какая у компании чистая прибыль за прошлый год?"
role_1 = "public_user" # Обычный работник

print(f"ПОЛЬЗОВАТЕЛЬ: {role_1}")
print(f"ЗАПРОС: '{query_1}'")
results_1 = secure_retriever(query_1, user_role=role_1, similarity_threshold=0.3, k=2)

print(f"НАЙДЕНО ДОКУМЕНТОВ: {len(results_1)}")
for res in results_1:
    print(f" -> [{res['category'].upper()}] (Сходство: {res['score']}) {res['title']}: {res['content'][:50]}...")
if not results_1:
    print(" -> [СИСТЕМА БЕЗОПАСНОСТИ]: Доступ запрещен или ничего не найдено.")

print("-" * 50)

# Эксперимент 2: Администратор задает тот же самый вопрос
# Ему система должна все показать
role_2 = "admin_user" # Руководитель / Админ

print(f"ПОЛЬЗОВАТЕЛЬ: {role_2}")
print(f"ЗАПРОС: '{query_1}'")
results_2 = secure_retriever(query_1, user_role=role_2, similarity_threshold=0.3, k=2)

print(f"НАЙДЕНО ДОКУМЕНТОВ: {len(results_2)}")
for res in results_2:
    print(f" -> [{res['category'].upper()}] (Сходство: {res['score']}) {res['title']}: {res['content']}")

print("-" * 50)

# Эксперимент 3: Влияние параметра Порога Сходства (Similarity Threshold)
# Ищем правила пропусков. Если порог слишком высокий (например, 0.8), мы можем ничего не найти
query_3 = "Как заходить в офис утром?"
print(f"ЗАПРОС: '{query_3}' с ВЫСОКИМ порогом (0.8)")
results_high_threshold = secure_retriever(query_3, user_role="public_user", similarity_threshold=0.8)
print(f" Найдено при пороге 0.8: {len(results_high_threshold)} док.")

print(f"ЗАПРОС: '{query_3}' с ОПТИМАЛЬНЫМ порогом (0.3)")
results_low_threshold = secure_retriever(query_3, user_role="public_user", similarity_threshold=0.3)
print(f" Найдено при пороге 0.3: {len(results_low_threshold)} док.")
for res in results_low_threshold:
    print(f"    -> Из документа '{res['title']}' (Сходство: {res['score']})")

Загрузка ИИ-модели эмбеддингов...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Индексация базы знаний (перевод текстов в векторы)...
База знаний готова к безопасным запросам!
ПОЛЬЗОВАТЕЛЬ: public_user
ЗАПРОС: 'Какая у компании чистая прибыль за прошлый год?'
НАЙДЕНО ДОКУМЕНТОВ: 0
 -> [СИСТЕМА БЕЗОПАСНОСТИ]: Доступ запрещен или ничего не найдено.
--------------------------------------------------
ПОЛЬЗОВАТЕЛЬ: admin_user
ЗАПРОС: 'Какая у компании чистая прибыль за прошлый год?'
НАЙДЕНО ДОКУМЕНТОВ: 1
 -> [SECRET] (Сходство: 0.613) Финансовый отчет (Конфиденциально): Секретный финансовый план на 2026 год. Чистая прибыль компании составила 150 миллионов рублей. Информация строго для руководства.
--------------------------------------------------
ЗАПРОС: 'Как заходить в офис утром?' с ВЫСОКИМ порогом (0.8)
 Найдено при пороге 0.8: 0 док.
ЗАПРОС: 'Как заходить в офис утром?' с ОПТИМАЛЬНЫМ порогом (0.3)
 Найдено при пороге 0.3: 1 док.
    -> Из документа 'Устав компании' (Сходство: 0.599)
